# Simple Writeup

## 1. Simple Strong Baseline: Ridge + LightGBM + Rain-3 + Nodewise Blend

This was the main tabular baseline, built from Ridge regression and LightGBM.

First, a joint autoregressive Ridge model was trained to predict water-level deltas for 1D and 2D nodes within each model. The Ridge model used lagged water levels, lagged rainfall features, and static node features.

Second, for each target `(model_id, node_type)`, three LightGBM models were trained for different rainfall regimes (`dry`, `mid`, and `wet`). During inference, these three LightGBM models were combined by rainfall-dependent weights, which is referred to here as the Rain-3 mixture.

Both Ridge and LightGBM predicted deltas autoregressively, and the next water level was obtained by adding the predicted delta to the previous step.

Finally, the Ridge and LightGBM predictions were combined by a per-node closed-form blend learned from out-of-fold predictions. This was not a simple average: each node could prefer Ridge or LightGBM differently.

This baseline worked well for Model 1 and also for Model 2 / 2D.

For Model 2 / 2D, this baseline was further improved by adding a small set of physics-inspired features to both Ridge and LightGBM. These features were kept simple and interpretable, and included:

- delta-by-static projections
- rain-by-static interactions
- 2D graph smoothness summaries
- 2D area-weighted mass summaries

However, this tabular approach was still weaker for Model 2 / 1D, so a different method was used for that target only.

## 2. Model 2 / 1D Only: Physics-Informed GNN with Graph Attention, Multi-Step Rollout, and Rain Gate

For Model 2 / 1D, a dedicated physics-informed GNN was used for water-level forecasting.

This model was developed from a public PI-GNN style example notebook and then adapted for this competition setting. The final Model 2 / 1D notebook combined the following ideas:

- graph attention instead of a simpler mean-style message passing variant
- multi-step rollout loss for more stable recursive forecasting
- a rain-conditioned gate ("RainGate") to modulate hidden states from rainfall context
- static node features

This model also predicted water-level deltas rather than absolute values, and recursive rollout was performed by adding the predicted delta to the previous state at each step.

Among the variants tried, deeper models were important in practice, and the final candidates used around 10 or more layers. It was also observed that `dropout = 0.0` worked better than `dropout = 0.1` in the final experiments, suggesting that a higher-capacity model was more effective for this target.

Shallower versions, such as roughly 3-layer and 5-layer variants, were also tried, and performance seemed to improve substantially as the model became deeper. The exact comparison numbers are not preserved in a clean table anymore, so the following figures should be read only as rough recollections rather than exact results: approximately `0.19` for the Ridge + LightGBM approach on Model 2 / 1D, around `0.09` for a shallower PI-GNN, and around `0.045` for the stronger deeper version.

## 3. Ensemble

We prepared three final ensemble styles:

1. target-wise single-best selection
2. type-wise closed-form blending
3. nodewise closed-form blending

Here, "single-best" simply means that only the source that appeared best from previous observations was used for each target, instead of mixing multiple sources.

In the end, the nodewise blend gave the best Public LB among the final submissions. In practice, adding too many weak or mismatched sources could hurt performance, so the ensemble was kept focused on sources that were already strong for the corresponding target.


# Materials

## Ensemble Notebooks

### No.1

`[submission LB 0.0166]`  
`[Ensemble] Nodewise Closed-Form (Event-Balanced) - Version 2`

https://www.kaggle.com/code/mtmrs1/ensemble-nodewise-closed-form-event-balanced?scriptVersionId=303753947

### No.2

`[submission LB 0.0167]`  
`[Ensemble] Typewise Closed-Form - Version 17`

https://www.kaggle.com/code/mtmrs1/ensemble-typewise-closed-form?scriptVersionId=303750250

### No.3

`[submission LB 0.0169]`  
`[Ensemble] Target Single-Best CV - Version 30`

https://www.kaggle.com/code/mtmrs1/ensemble-target-single-best-cv?scriptVersionId=303640758

### Note
submission No.1 uses all predictions listed below for ensembling.

submission No.2 uses all predictions listed below for ensembling, except for Model2 2D(best only for Model2 2D).

submission No.3 uses only best predictions for each type (model / 1D2D).

## Model Runs

### Model 1 (1D+2D)

**best**

Ridge + LightGBM Rain-3 w/ Static Feats (Model1) - Version 2

https://www.kaggle.com/code/mtmrs1/ridge-lightgbm-rain-3-w-static-feats-model1

### Model 2 (1D)

**best**

PI-GNN GraphAttention + Rollout + RainGate L10 Do0 - Version 1

https://www.kaggle.com/datasets/mtmrs1/pi-gnn-graphattention-rollout-raingate-l10-do0

**Used for Ensemble**

PI-GNN GraphAttention + Rollout + RainGate L10 - Version 1

https://www.kaggle.com/datasets/mtmrs1/pi-gnn-graphattention-rollout-raingate-l10

PI-GNN GraphAttention + Rollout + RainGate L15 Do0 - Version 1

https://www.kaggle.com/datasets/mtmrs1/pi-gnn-graphattention-rollout-raingate-l15-do0

PI-GNN with G-Att, Rollout, R-Gate, 12-L, 0.0 Drop - Version 2

https://www.kaggle.com/code/mtmrs1/pi-gnn-with-g-att-rollout-r-gate-12-l-0-0-drop

### Model 2 (2D)

**best**

Ridge + LGBM Rain-3 w/ Static + PI Feats (M2, 2D) - Version 1

https://www.kaggle.com/code/mtmrs1/ridge-lgbm-rain-3-w-static-pi-feats-m2-2d

**Used for Ensemble**

Ridge + LightGBM Rain-3 w/ Static Feats (M2,2D) - Version 1

https://www.kaggle.com/code/mtmrs1/ridge-lightgbm-rain-3-w-static-feats-m2-2d

## Dataset

Just a copy of the official data revised

urban-flood-bench-re

https://www.kaggle.com/datasets/mtmrs1/urban-flood-bench-re
